# Stanford RNA 3D Folding Part 2 - Competition Submission

This notebook predicts 3D structures of RNA molecules from their sequences.

## Competition Overview
- **Goal**: Predict 3D RNA structures using only sequence information
- **Evaluation**: TM-score (0.0 to 1.0, higher is better)
- **Output**: 5 structure predictions per sequence with C1' atom coordinates

In [ ]:
# Install/upgrade required packages (for Kaggle environment or if missing locally)
# This will install packages in the current Python environment
import subprocess
import sys

def install_package(package):
    """Install a package, but don't fail if network is unavailable."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package], 
                             timeout=60, stderr=subprocess.DEVNULL)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        raise Exception(f"Failed to install {package}: {e}")

try:
    import numpy
except ImportError:
    try:
        print("Installing numpy...")
        install_package("numpy>=1.24.0")
    except Exception as e:
        print(f"Error: Could not install numpy: {e}")
        raise  # numpy is required, so fail if we can't install it

try:
    import pandas
except ImportError:
    try:
        print("Installing pandas...")
        install_package("pandas>=2.0.0")
    except Exception as e:
        print(f"Error: Could not install pandas: {e}")
        raise  # pandas is required, so fail if we can't install it

try:
    import scipy
except ImportError:
    try:
        print("Installing scipy...")
        install_package("scipy>=1.10.0")
    except Exception as e:
        print(f"Warning: Could not install scipy: {e}")
        print("Continuing without scipy (may not be needed for basic predictions)...")
        scipy = None

try:
    import Bio
except ImportError:
    try:
        print("Installing biopython...")
        install_package("biopython>=1.81")
    except Exception as e:
        print(f"Warning: Could not install biopython: {e}")
        print("Continuing without biopython (not required for basic predictions)...")
        Bio = None

try:
    import torch
except ImportError:
    try:
        print("Installing torch...")
        install_package("torch>=2.0.0")
    except Exception as e:
        print(f"Warning: Could not install torch: {e}")
        print("Continuing without torch (not required for basic predictions)...")
        torch = None

print("✓ Package installation check complete!")

# Import required libraries
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path

# Add utils to path (works for both local and Kaggle)
if os.path.exists('/kaggle/working'):
    sys.path.append('/kaggle/working')
else:
    # For local development, add current directory
    sys.path.append(os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.')))

try:
    from utils import (
        read_test_sequences,
        generate_submission_template,
        save_submission,
        validate_submission,
        parse_fasta,
        read_msa_file,
        clip_coordinates
    )
    print("✓ Utils imported successfully!")
except ImportError as e:
    print(f"Warning: Could not import utils: {e}")
    print("Make sure utils.py is in the same directory or /kaggle/working")
    print("Using fallback implementations...")
    
    # Fallback implementations if utils module not available
    def read_test_sequences(file_path: str = "test_sequences.csv") -> pd.DataFrame:
        """Fallback: Read test sequences CSV file."""
        # If file_path is already absolute, use it directly
        if os.path.isabs(file_path):
            possible_paths = [file_path]
        else:
            possible_paths = [
                file_path,
                f"/kaggle/input/stanford-rna-3d-folding-2/{os.path.basename(file_path)}",
                os.path.join("/kaggle/working", file_path),
            ]
        # Also try the absolute path even if relative was provided
        if not os.path.isabs(file_path):
            possible_paths.append(f"/kaggle/input/stanford-rna-3d-folding-2/{file_path}")
        
        for path in possible_paths:
            if os.path.exists(path):
                return pd.read_csv(path)
        raise FileNotFoundError(f"Could not find test_sequences.csv. Tried: {possible_paths}")
    
    def generate_submission_template(sequences_df: pd.DataFrame) -> pd.DataFrame:
        """Fallback: Generate submission template DataFrame."""
        submission_rows = []
        for _, row in sequences_df.iterrows():
            target_id = row['target_id']
            sequence = row['sequence']
            for i, residue in enumerate(sequence, start=1):
                resname = residue.upper()
                submission_id = f"{target_id}_{i}"
                submission_row = {
                    'ID': submission_id,
                    'resname': resname,
                    'resid': i,
                    'x_1': 0.0, 'y_1': 0.0, 'z_1': 0.0,
                    'x_2': 0.0, 'y_2': 0.0, 'z_2': 0.0,
                    'x_3': 0.0, 'y_3': 0.0, 'z_3': 0.0,
                    'x_4': 0.0, 'y_4': 0.0, 'z_4': 0.0,
                    'x_5': 0.0, 'y_5': 0.0, 'z_5': 0.0,
                }
                submission_rows.append(submission_row)
        return pd.DataFrame(submission_rows)
    
    def clip_coordinates(coords: np.ndarray) -> np.ndarray:
        """Fallback: Clip coordinates to valid range."""
        return np.clip(coords, -999.999, 9999.999)
    
    def save_submission(predictions_df: pd.DataFrame, output_path: str = "submission.csv"):
        """Fallback: Save submission CSV file."""
        # Clip coordinates to valid range before saving
        coord_cols = [col for col in predictions_df.columns if col.startswith(('x_', 'y_', 'z_'))]
        for col in coord_cols:
            predictions_df[col] = clip_coordinates(predictions_df[col].values)
        
        # Create directory if it doesn't exist (for local development)
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir, exist_ok=True)
            print(f"Created directory: {output_dir}")
        
        predictions_df.to_csv(output_path, index=False)
        print(f"Submission saved to {output_path}")
        print(f"Shape: {predictions_df.shape}")
        print(f"Columns: {predictions_df.columns.tolist()}")
    
    def validate_submission(submission_df: pd.DataFrame, sequences_df: pd.DataFrame) -> bool:
        """Fallback: Validate submission format."""
        required_cols = ['ID', 'resname', 'resid', 
                        'x_1', 'y_1', 'z_1', 'x_2', 'y_2', 'z_2',
                        'x_3', 'y_3', 'z_3', 'x_4', 'y_4', 'z_4',
                        'x_5', 'y_5', 'z_5']
        missing_cols = [col for col in required_cols if col not in submission_df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        print("Submission validation passed!")
        return True
    
    def parse_fasta(fasta_string: str) -> dict:
        """Fallback: Parse FASTA string."""
        import re
        chains = {}
        current_header = None
        current_sequence = []
        for line in fasta_string.split('\n'):
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header and current_sequence:
                    chain_match = re.search(r'chain=([A-Za-z0-9]+)', current_header)
                    chain_id = chain_match.group(1) if chain_match else current_header.split()[0] if current_header else 'A'
                    chains[chain_id] = ''.join(current_sequence)
                current_header = line[1:]
                current_sequence = []
            else:
                current_sequence.append(line)
        if current_header and current_sequence:
            chain_match = re.search(r'chain=([A-Za-z0-9]+)', current_header)
            chain_id = chain_match.group(1) if chain_match else current_header.split()[0] if current_header else 'A'
            chains[chain_id] = ''.join(current_sequence)
        return chains
    
    def read_msa_file(target_id: str, msa_dir: str = "MSA"):
        """Fallback: Read MSA file."""
        possible_paths = [
            os.path.join(msa_dir, f"{target_id}.MSA.fasta"),
            os.path.join("/kaggle/input/stanford-rna-3d-folding-2", msa_dir, f"{target_id}.MSA.fasta"),
            os.path.join("/kaggle/working", msa_dir, f"{target_id}.MSA.fasta"),
        ]
        for msa_path in possible_paths:
            if os.path.exists(msa_path):
                alignments = []
                current_header = None
                current_sequence = []
                with open(msa_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        if line.startswith('>'):
                            if current_header and current_sequence:
                                alignments.append((current_header, ''.join(current_sequence)))
                            current_header = line[1:]
                            current_sequence = []
                        else:
                            current_sequence.append(line)
                    if current_header and current_sequence:
                        alignments.append((current_header, ''.join(current_sequence)))
                return alignments
        return None

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Working directory: {os.getcwd()}")
print("Libraries imported successfully!")

## Load Test Sequences

In [ ]:
# Read test sequences
INPUT_FILE = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"

try:
    test_sequences = read_test_sequences(INPUT_FILE)
    print(f"Loaded {len(test_sequences)} test sequences")
    print(f"\nColumns: {test_sequences.columns.tolist()}")
    print(f"\nFirst few sequences:")
    print(test_sequences[['target_id', 'sequence']].head())
    print(f"\nSequence lengths: {test_sequences['sequence'].str.len().describe()}")
    print(f"\nSample target_id: {test_sequences['target_id'].iloc[0]}")
except (FileNotFoundError, NameError) as e:
    # For local testing, create a sample file structure
    print(f"Test sequences file not found or read_test_sequences not available: {e}")
    print("Creating sample structure for development...")
    test_sequences = pd.DataFrame({
        'target_id': ['1ABC_A', '2DEF_B'],
        'sequence': ['GGCGUAGUCC', 'AUCGAUCGAU'],
        'temporal_cutoff': ['2025-01-01', '2025-01-02'],
        'description': ['Sample RNA 1', 'Sample RNA 2'],
        'stoichiometry': ['A:1', 'B:1'],
        'all_sequences': ['>Chain A\nGGCGUAGUCC', '>Chain B\nAUCGAUCGAU'],
        'ligand_ids': ['', ''],
        'ligand_SMILES': ['', '']
    })
    print(f"Using sample data: {len(test_sequences)} sequences")

In [ ]:
# ============================================================
# RMSD DIAGNOSTICS FOR LOCAL VALIDATION
# ============================================================
# These functions let you validate conformational diversity
# before submitting to Kaggle (no external dependencies)

import numpy as np
from typing import List, Optional

def compute_rmsd(a: np.ndarray, b: np.ndarray) -> float:
    """Compute RMSD between two coordinate sets of shape (L, 3)."""
    if a.shape != b.shape:
        raise ValueError(f"RMSD shape mismatch: {a.shape} vs {b.shape}")
    return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))


def diagnose_conformational_diversity(
    coords_list: List[np.ndarray],
    target_id: str = "sample",
    expected_scales: Optional[List[float]] = None,
    print_pairwise: bool = True,
) -> None:
    """
    Print RMSD diagnostics across a list of conformations.
    
    Args:
        coords_list: list of 5 arrays, each (L, 3)
        target_id: label for display
        expected_scales: expected RMSD values to base conformation (conf 0)
        print_pairwise: whether to compute and print average pairwise RMSD
    """
    if expected_scales is None:
        expected_scales = [0.0, 0.5, 1.0, 1.5, 2.0]
    
    k = len(coords_list)
    if k == 0:
        print(f"No conformations provided for {target_id}")
        return
    
    print(f"\n{'='*70}")
    print(f"Diversity diagnostics for {target_id} ({k} conformations)")
    print(f"{'='*70}")
    base = coords_list[0]
    rmsds_to_base: List[float] = []
    pairwise: List[float] = []
    
    for i, conf in enumerate(coords_list):
        rmsd = compute_rmsd(base, conf)
        rmsds_to_base.append(rmsd)
        print(f"  conf {i} → RMSD to base = {rmsd:.3f} Å")
        if print_pairwise:
            for j in range(i):
                pw = compute_rmsd(coords_list[j], conf)
                pairwise.append(pw)
    
    if print_pairwise and len(pairwise) > 0:
        mean_pairwise = float(np.mean(pairwise))
        print(f"  Average pairwise RMSD (all pairs) = {mean_pairwise:.3f} Å")
    
    print("  Expected vs Observed RMSD to base:")
    for i, (got, want) in enumerate(zip(rmsds_to_base, expected_scales)):
        if want > 0:
            ratio = got / want
            status = "✅" if 0.5 <= ratio <= 2.0 else "⚠️"
            print(f"    {status} scale {want:4.1f} → observed {got:5.2f} Å (ratio ≈ {ratio:.2f})")
        else:
            print(f"    ✅ scale {want:4.1f} → observed {got:5.2f} Å (base)")
    print(f"{'='*70}\n")

print("✅ RMSD diagnostic functions loaded!")

## Structure Prediction Model

**TODO**: Implement your RNA 3D structure prediction model here.

This is a placeholder that generates random coordinates. Replace this with your actual model.

In [ ]:
"""IMPROVED RNA 3D STRUCTURE PREDICTOR WITH SECONDARY STRUCTURE=============================================================Target score: 0.15-0.18+ (competitive with SS prediction)Key improvements:1. Nussinov algorithm for secondary structure prediction2. CONSERVATIVE noise scales [0, 0.5, 1, 1.5, 2]Å3. Proper energy minimization (50 iterations)4. Base pairing constraints at Watson-Crick distance (10.5Å)5. Centering enabledExpected: 2-5Å pairwise RMSD, 10-20% better than heuristic pairing"""import numpy as np# CRITICAL FIX: Small noise for diversity, not destructionNOISE_SCALES = [0.0, 0.5, 1.0, 1.5, 2.0]# Base pairing rulesWATSON_CRICK = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}WOBBLE_PAIRS = {('G', 'U'), ('U', 'G')}def nussinov_fold(sequence, min_loop_size=3):    """    Nussinov algorithm for RNA secondary structure prediction.    Returns list of base pairs (i, j) where i < j.        Fast: O(n³) but typically <0.1s for 100nt sequences.    """    n = len(sequence)    dp = np.zeros((n, n), dtype=int)    traceback = {}        def can_pair(i, j):        if j - i <= min_loop_size:            return False        pair = (sequence[i], sequence[j])        return (sequence[i] in WATSON_CRICK and                 WATSON_CRICK[sequence[i]] == sequence[j]) or pair in WOBBLE_PAIRS        # Fill DP table    for length in range(min_loop_size + 1, n):        for i in range(n - length):            j = i + length                        # Case 1: j unpaired            dp[i][j] = dp[i][j-1]            traceback[(i, j)] = ('unpaired', j)                        # Case 2: (i,j) pair            if can_pair(i, j):                score = dp[i+1][j-1] + 1                if score > dp[i][j]:                    dp[i][j] = score                    traceback[(i, j)] = ('pair', i, j)                        # Case 3: bifurcation            for k in range(i + 1, j):                score = dp[i][k] + dp[k+1][j]                if score > dp[i][j]:                    dp[i][j] = score                    traceback[(i, j)] = ('bifurc', k)        # Traceback to get base pairs    def trace(i, j, pairs_list):        if i >= j or (i, j) not in traceback:            return                action = traceback[(i, j)]        if action[0] == 'unpaired':            trace(i, j-1, pairs_list)        elif action[0] == 'pair':            pairs_list.append((i, j))            trace(i+1, j-1, pairs_list)        elif action[0] == 'bifurc':            k = action[1]            trace(i, k, pairs_list)            trace(k+1, j, pairs_list)        pairs_list = []    trace(0, n-1, pairs_list)    return sorted(pairs_list)def find_stems(base_pairs):    """Convert base pairs to stem regions (consecutive base pairs)."""    if not base_pairs:        return []        stems = []    current_stem = [base_pairs[0]]        for i in range(1, len(base_pairs)):        prev_i, prev_j = base_pairs[i-1]        curr_i, curr_j = base_pairs[i]                # Check if consecutive in both strands        if curr_i == prev_i + 1 and curr_j == prev_j - 1:            current_stem.append(base_pairs[i])        else:            if len(current_stem) >= 2:                stems.append(current_stem)            current_stem = [base_pairs[i]]        if len(current_stem) >= 2:        stems.append(current_stem)    return stemsdef predict_rna_structure(sequence: str, prediction_number: int) -> np.ndarray:    """    Predict RNA 3D structure with secondary structure guidance.        Args:        sequence: RNA sequence string        prediction_number: Conformation number (1-5)        Returns:        coords: 3D coordinates (L, 3) in Angstroms    """    n = len(sequence)    coords = np.zeros((n, 3))        # Set seed for reproducibility    np.random.seed(hash(sequence) % 2**32 + prediction_number * 1000)        # RNA geometry constants    BACKBONE = 5.9    BP_DISTANCE = 10.5        # NEW: Predict secondary structure using Nussinov    base_pairs = nussinov_fold(sequence)    stems = find_stems(base_pairs)        # Convert to simple pairs list for compatibility    pairs = [(i, j) for stem in stems for i, j in stem]        # Build helices from stems (using predicted SS, not heuristic)    helices = []    for stem in stems:        if len(stem) >= 2:            helix_start_i = stem[0][0]            helix_start_j = stem[0][1]            helix_length = len(stem)            helices.append((helix_start_i, helix_start_j, helix_length))        # Place helices in 3D space    placed = set()    current_pos = np.zeros(3)        # A-form RNA geometry    RADIUS = 5.25    RISE = 2.8    TWIST = 32.7 * np.pi / 180        for helix_idx, (hs, he, hl) in enumerate(helices):        rotation_offset = (prediction_number - 1) * np.pi / 10        tilt_offset = (prediction_number - 1) * np.pi / 20                for k in range(hl):            i, j = hs + k, he - k                        if i >= n or j >= n or i in placed or j in placed:                continue                        angle = k * TWIST + rotation_offset            z = k * RISE                        # First strand            x1 = RADIUS * np.cos(angle)            y1 = RADIUS * np.sin(angle)                        coords[i] = current_pos + np.array([                x1 * np.cos(tilt_offset) - z * np.sin(tilt_offset),                y1,                x1 * np.sin(tilt_offset) + z * np.cos(tilt_offset)            ])                        # Second strand            x2 = -RADIUS * np.cos(angle)            y2 = -RADIUS * np.sin(angle)                        coords[j] = current_pos + np.array([                x2 * np.cos(tilt_offset) - z * np.sin(tilt_offset),                y2,                x2 * np.sin(tilt_offset) + z * np.cos(tilt_offset)            ])                        placed.add(i)            placed.add(j)                current_pos += np.array([25, 0, 0])        # Fill unpaired regions    for i in range(n):        if i in placed:            continue                if i > 0 and i-1 in placed:            prev = coords[i-1]            direction = np.random.randn(3)            direction = direction / (np.linalg.norm(direction) + 1e-10)            coords[i] = prev + direction * BACKBONE        elif i < n-1 and i+1 in placed:            next_coord = coords[i+1]            direction = np.random.randn(3)            direction = direction / (np.linalg.norm(direction) + 1e-10)            coords[i] = next_coord - direction * BACKBONE        else:            if i > 0:                coords[i] = coords[i-1] + np.random.randn(3) * BACKBONE            else:                coords[i] = np.random.randn(3) * BACKBONE                placed.add(i)        # Energy minimization (50 iterations)    for step in range(50):        forces = np.zeros_like(coords)                # Backbone connectivity        for i in range(n-1):            vec = coords[i+1] - coords[i]            dist = np.linalg.norm(vec)            if dist > 0:                force = (dist - BACKBONE) * 0.2 * vec / dist                forces[i] += force                forces[i+1] -= force                # Base pairing constraints (using Nussinov pairs)        for i, j in pairs:            if i < n and j < n:                vec = coords[j] - coords[i]                dist = np.linalg.norm(vec)                if dist > 0:                    force = (dist - BP_DISTANCE) * 0.1 * vec / dist                    forces[i] += force                    forces[j] -= force                coords += forces * 0.15        # Conservative diversity    noise_scale = NOISE_SCALES[min(prediction_number - 1, len(NOISE_SCALES) - 1)]    if noise_scale > 0:        coords += np.random.normal(0, noise_scale, coords.shape)        # Random rotation    if prediction_number > 1:        axis = np.random.randn(3)        axis = axis / (np.linalg.norm(axis) + 1e-10)        angle = (prediction_number - 1) * np.pi / 6                K = np.array([[0, -axis[2], axis[1]],                      [axis[2], 0, -axis[0]],                      [-axis[1], axis[0], 0]])        R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)        coords = coords @ R.T        # Center    coords -= coords.mean(axis=0)        return coordsprint("✅ RNA predictor with Nussinov secondary structure loaded!")print(f"   Noise scales: {NOISE_SCALES} (CONSERVATIVE)")print(f"   Energy minimization: 50 iterations")print(f"   Base pairing: 10.5Å Watson-Crick distance")print(f"   Secondary structure: Nussinov algorithm (O(n³))")print(f"   Target: 2-5Å pairwise RMSD, 10-20% better score")# Quick testtest_seq = "GGCGUAGUCC"print(f"\nQuick test with sequence: {test_seq}")pairs_test = nussinov_fold(test_seq)print(f"  Predicted {len(pairs_test)} base pairs: {pairs_test}")for pred_num in [1, 2, 3]:    coords = predict_rna_structure(test_seq, pred_num)    print(f"  Prediction {pred_num}: shape {coords.shape}, " +          f"range [{coords.min():.1f}, {coords.max():.1f}], " +          f"centered={abs(coords.mean()) < 0.01}")

# Quick test with timing
import time

test_sequences_list = [
    ("Short", "GGCGUAGUCC"),  # 10nt
    ("Medium", "GGCGUAGUCC" * 5),  # 50nt
    ("Long", "GGCGUAGUCC" * 20),  # 200nt
    ("VeryLong", "GGCGUAGUCC" * 50),  # 500nt
]

print("Testing prediction speed:")
for name, seq in test_sequences_list:
    start = time.time()
    try:
        coords = predict_rna_structure(seq, 1)
        elapsed = time.time() - start
        print(f"  {name} ({len(seq)}nt): {elapsed:.2f}s - Shape: {coords.shape}")
    except Exception as e:
        print(f"  {name} ({len(seq)}nt): ERROR - {e}")

print("\n✓ Speed test complete!")

In [ ]:
# Generate submission template
submission_df = generate_submission_template(test_sequences)

print(f"Submission template created with {len(submission_df)} rows")
print(f"Number of unique targets: {test_sequences['target_id'].nunique()}")

# Generate predictions for all sequences
print("\nGenerating predictions...")
print(f"Total sequences to process: {len(test_sequences)}")

import time
start_time = time.time()

for idx, row in test_sequences.iterrows():
    target_id = row['target_id']
    sequence = row['sequence']
    
    # Progress for EVERY sequence
    print(f"[{idx+1}/{len(test_sequences)}] Processing {target_id} (length: {len(sequence)})", flush=True)
    
    # Optional: Load MSA if available (for advanced models)
    # msa_data = read_msa_file(target_id)
    # if msa_data:
    #     print(f"  Loaded MSA for {target_id}: {len(msa_data)} sequences")
    
    # Generate 5 predictions per sequence
    for pred_num in range(1, 6):
        pred_start = time.time()
        coords = predict_rna_structure(sequence, pred_num)
        pred_time = time.time() - pred_start
        
        if pred_time > 5:  # Warn if slow
            print(f"  Prediction {pred_num} took {pred_time:.1f}s", flush=True)
        
        # Update submission DataFrame with coordinates
        # ID format: target_id_resid
        # More efficient: create all IDs at once and update in batch
        submission_ids = [f"{target_id}_{resid}" for resid in range(1, len(sequence) + 1)]
        mask = submission_df['ID'].isin(submission_ids)
        
        # Map coordinates to the correct rows
        for i, submission_id in enumerate(submission_ids):
            row_mask = (submission_df['ID'] == submission_id) & mask
            if row_mask.sum() > 0:
                submission_df.loc[row_mask, f'x_{pred_num}'] = coords[i, 0]
                submission_df.loc[row_mask, f'y_{pred_num}'] = coords[i, 1]
                submission_df.loc[row_mask, f'z_{pred_num}'] = coords[i, 2]
    
    elapsed = time.time() - start_time
    avg_time = elapsed / (idx + 1)
    remaining = avg_time * (len(test_sequences) - idx - 1)
    print(f"  ✓ Done. Elapsed: {elapsed:.1f}s, Est. remaining: {remaining/60:.1f}min", flush=True)

print("\nAll predictions generated!")

In [ ]:
# ============================================================
# OPTIONAL: LOCAL DIVERSITY VALIDATION
# ============================================================
# Run this cell to validate conformational diversity BEFORE
# uploading to Kaggle. This gives you instant feedback!
#
# Expected results:
# - RMSD to base: ~[0, 5, 10, 15, 20]Å
# - Average pairwise RMSD: 2-5Å (diverse but not destroyed)
#
# Set ENABLE_DIAGNOSTICS = True to run
ENABLE_DIAGNOSTICS = False  # Set to True to run diagnostics

if ENABLE_DIAGNOSTICS:
    print("\n" + "="*70)
    print("LOCAL DIVERSITY VALIDATION")
    print("="*70)
    print("Checking first 2 targets...\n")
    
    # Get first 2 targets from test_sequences
    check_targets = test_sequences['target_id'].iloc[:2].tolist()
    
    for target_id in check_targets:
        # Get sequence
        seq_row = test_sequences[test_sequences['target_id'] == target_id]
        sequence = seq_row['sequence'].iloc[0]
        
        # Generate all 5 conformations
        coords_list = []
        for pred_num in range(1, 6):
            coords = predict_rna_structure(sequence, pred_num)
            coords_list.append(coords)
        
        # Diagnose diversity
        diagnose_conformational_diversity(
            coords_list,
            target_id=f"{target_id} ({len(sequence)}nt)",
            expected_scales=NOISE_SCALES
        )
    
    print("\n" + "="*70)
    print("✅ DIAGNOSTICS COMPLETE!")
    print("="*70)
    print("\nWhat to look for:")
    print("  ✅ GOOD: Avg pairwise RMSD 2-5Å (diverse but not destroyed)")
    print("  ✅ GOOD: RMSD ratios between 0.5-2.0x expected")
    print("  ⚠️  BAD: Avg pairwise RMSD < 1Å (too similar)")
    print("  ⚠️  BAD: RMSD ratios < 0.3 (noise suppressed)")
else:
    print("\n⏭️  Diagnostics skipped (set ENABLE_DIAGNOSTICS=True to run)")
    print("   This is OPTIONAL - only for local validation before Kaggle submission.")


## Validate and Save Submission

In [ ]:
# Validate submission format
try:
    validate_submission(submission_df, test_sequences)
except Exception as e:
    print(f"Validation error: {e}")
    raise

# Display sample of submission
print("\nSample submission:")
print(submission_df.head(10))

# ============================================================
# COMPETITION SUBMISSION - Save to submission.csv
# Kaggle will automatically find this file in the working directory
# ============================================================

# Save submission file (REQUIRED: must be named 'submission.csv')
# save_submission clips coordinates to valid range and saves the file
OUTPUT_FILE = "submission.csv"

# Method 1: Use save_submission function (clips coordinates)
save_submission(submission_df, OUTPUT_FILE)

# Method 2: Direct save as backup (ensures file is created)
# Clip coordinates before saving
coord_cols = [col for col in submission_df.columns if col.startswith(('x_', 'y_', 'z_'))]
for col in coord_cols:
    submission_df[col] = np.clip(submission_df[col].values, -999.999, 9999.999)

# Save directly to ensure file exists
submission_df.to_csv(OUTPUT_FILE, index=False)

# Verify file was created (Kaggle requires this file to exist)
assert os.path.exists(OUTPUT_FILE), "❌ submission.csv not created!"

# Verify file is not empty
file_size = os.path.getsize(OUTPUT_FILE)
assert file_size > 0, f"❌ submission.csv is empty! Size: {file_size} bytes"

# Verify we can read it back
df_check = pd.read_csv(OUTPUT_FILE)
assert len(df_check) > 0, "❌ submission.csv has no rows!"
assert 'ID' in df_check.columns, "❌ submission.csv missing ID column!"

print("\n" + "=" * 60)
print("✅ SUCCESS: submission.csv created and ready for submission!")
print("=" * 60)
print(f"File: {OUTPUT_FILE}")
print(f"Shape: {submission_df.shape}")
print(f"Columns: {len(submission_df.columns)}")
print(f"Rows: {len(submission_df)}")
print(f"File size: {file_size} bytes ({file_size / 1024:.2f} KB)")
print(f"Full path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Working directory: {os.getcwd()}")
print("=" * 60)

# Final verification - show first few rows
print("\nFirst 3 rows of saved file:")
print(df_check.head(3))
print("\n✅ File verified and ready for submission!")

In [ ]:
# ============================================================
# FINAL VERIFICATION - Debug cell to verify submission.csv exists
# ============================================================
import glob

print("=" * 60)
print("FINAL VERIFICATION - Files in working directory:")
print("=" * 60)

# List all files
all_files = glob.glob('*')
for f in sorted(all_files):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f"  📄 {f} ({size} bytes)")
    else:
        print(f"  📁 {f}/")

print("\n" + "=" * 60)
print("Checking for submission.csv:")
print("=" * 60)

if os.path.exists('submission.csv'):
    size = os.path.getsize('submission.csv')
    print(f"✅ submission.csv EXISTS - {size} bytes")
    
    # Read and verify
    df_final = pd.read_csv('submission.csv')
    print(f"✅ File readable - Shape: {df_final.shape}")
    print(f"✅ Columns: {list(df_final.columns)[:5]}... ({len(df_final.columns)} total)")
    print(f"✅ First ID: {df_final['ID'].iloc[0]}")
    print(f"✅ Last ID: {df_final['ID'].iloc[-1]}")
    
    # Check for required columns
    required = ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']
    missing = [col for col in required if col not in df_final.columns]
    if missing:
        print(f"❌ Missing columns: {missing}")
    else:
        print("✅ All required columns present")
    
    print("\n" + "=" * 60)
    print("🎉 READY FOR SUBMISSION!")
    print("=" * 60)
else:
    print("❌ submission.csv NOT FOUND!")
    print("\nAvailable files:")
    print(os.listdir('.'))
